In [4]:
import os
from typing import List, Optional, Generator
import heapq
import torch
from owlapy.class_expression import OWLClassExpression, OWLThing
from owlapy.utils import get_expression_length
from ontolearn.heuristics import CeloeBasedReward
from ontolearn.learners import TDL
from ontolearn.triple_store import TripleStore
from ontolearn.knowledge_base import KnowledgeBase
from ontolearn.learning_problem import PosNegLPStandard
from owlapy.owl_individual import OWLNamedIndividual
from ontolearn.utils import read_csv
from owlapy import owl_expression_to_sparql, owl_expression_to_dl, owl_expression_to_sparql_with_confusion_matrix
from dicee.executer import Execute
from dicee.config import Namespace
from ontolearn.search import RL_State
from ontolearn.utils.static_funcs import compute_f1_score_from_confusion_matrix, compute_f1_score
from ontolearn.refinement_operators import LengthBasedRefinement

In [5]:
# set working directory
os.chdir("..")
os.chdir("..")
os.getcwd() 

'D:\\PycharmProjects\\Ontolearn'

In [6]:
kb = KnowledgeBase(path="KGs/Family/father.owl")

In [7]:
lp = PosNegLPStandard(pos={OWLNamedIndividual("http://example.com/father#stefan")},
                      neg={OWLNamedIndividual("http://example.com/father#heinz"),
                           OWLNamedIndividual("http://example.com/father#anna"),
                           OWLNamedIndividual("http://example.com/father#michelle")})

In [8]:
pos = lp.pos
neg = lp.neg
print("Positive examples:", [str(ind) for ind in pos])
print("Negative examples:", [str(ind) for ind in neg])

Positive examples: ["OWLNamedIndividual(IRI('http://example.com/father#', 'stefan'))"]
Negative examples: ["OWLNamedIndividual(IRI('http://example.com/father#', 'heinz'))", "OWLNamedIndividual(IRI('http://example.com/father#', 'anna'))", "OWLNamedIndividual(IRI('http://example.com/father#', 'michelle'))"]


In [8]:
def get_kb_embeddings(model = 'Keci', path_single_kg = "KGs/Family/father.owl",
                      path_to_store_single_run = "KGs/Family/embeddings_father",
                      num_epochs = 100,
                      embedding_dim = 256):
    args = Namespace()
    args.model = model
    args.path_single_kg = path_single_kg
    args.path_to_store_single_run = path_to_store_single_run
    args.num_epochs = num_epochs
    args.embedding_dim = embedding_dim
    args.batch_size = 64
    args.backend = "rdflib"
    args.eval_model = "train_val_test"
    args.n_epochs_eval_model = "train_val_test"
    args.trainer = "MO"
    args.save_embeddings_as_csv = True
    args.threshold = 200
    reports = Execute(args).start()
    return reports 

In [9]:
path_embeddings = "KGs/Family/embeddings_father/Keci_entity_embeddings.csv"

In [31]:
#get_kb_embeddings()

Seed set to 0

   | Name                             | Type              | Params | Mode 
--------------------------------------------------------------------------------
0  | loss                             | BCEWithLogitsLoss | 0      | train
1  | normalize_head_entity_embeddings | IdentityClass     | 0      | train
2  | normalize_relation_embeddings    | IdentityClass     | 0      | train
3  | normalize_tail_entity_embeddings | IdentityClass     | 0      | train
4  | hidden_normalizer                | IdentityClass     | 0      | train
5  | input_dp_ent_real                | Dropout           | 0      | train
6  | input_dp_rel_real                | Dropout           | 0      | train
7  | hidden_dropout                   | Dropout           | 0      | train
8  | entity_embeddings                | Embedding         | 4.4 K  | train
9  | relation_embeddings              | Embedding         | 3.6 K  | train
10 | q_coefficients                   | Embedding         | 1      | train
----

Start time:2025-11-22 14:48:55.736406
*** Read or Load Knowledge Graph  ***
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to obtain index...
Creating a mapping from entities to integer indexes...
preprocess_with_pandas took 0.0045 seconds | Current Memory Usage  236.84 in MB
Submit er-vocab, re-vocab, and ee-vocab via  ProcessPoolExecutor...
Preprocessing took: 0.032 seconds

------------------- Description of Dataset KGs/Family/father.owl -------------------
Number of entities:17
Number of relations:14
Number of triples on train set:58
Number of triples on valid set:0
Number of triples on test set:0
Entity Index:0.00000 in GB
Relation Index:0.00000 in GB

# of CPUs:16 | # of GPUs:0 | # of CPUs for dataloader:0
------------------- Train -------------------
Initializing TorchTrainer CPU Trainer...	Took 0.0049 secs | Current Memory Usage  236.88 in MB
Initializing Model...	Took 0.0009 secs | Current Memory Usage  236.88 in MB
Initializ

Epoch:100: 100%|██████████| 100/100 [00:00<00:00, 108.93it/s, loss_step=0.00000, loss_epoch=0.00000]


Training Runtime: 0.939 seconds.

*** Save Trained Model ***
Saving entity embeddings...
Saving relation embeddings...
Took 0.0576 secs | Current Memory Usage  236.95 in MB
Total Runtime: 1.032 seconds
Evaluate Keci on Train set: Evaluate Keci on Train set
{'H@1': 1.0, 'H@3': 1.0, 'H@10': 1.0, 'MRR': 1.0}
Total Runtime: 9.966 seconds


{'num_train_triples': 58,
 'num_entities': 17,
 'num_relations': 14,
 'max_length_subword_tokens': None,
 'runtime_kg_loading': 0.03214144706726074,
 'EstimatedSizeMB': 0.007569313049316406,
 'NumParam': 7937,
 'path_experiment_folder': 'KGs/Family/embeddings_father',
 'Runtime': 9.965751886367798,
 'Train': {'H@1': 1.0, 'H@3': 1.0, 'H@10': 1.0, 'MRR': 1.0}}

In [10]:
df_embeddings = read_csv(path_embeddings).astype('float32')

In [11]:
def get_embeddings_individuals(individuals: List[str]) -> torch.FloatTensor:
    embedding_dim = 256
    assert isinstance(individuals, list)
    if len(individuals) == 0:
        emb = torch.zeros(1, 1, embedding_dim)
    else:
        if df_embeddings is not None:
            assert isinstance(individuals[0], str)
            emb = torch.mean(torch.from_numpy(df_embeddings.loc[individuals].values), dim=0)
            emb = emb.view(1, 1, embedding_dim)
        else:
            emb = torch.zeros(1, 1, embedding_dim)
    return emb

self.emb_pos = self.get_embeddings_individuals(individuals=[i.str for i in self.pos]).to(self.device)
self.emb_neg = self.get_embeddings_individuals(individuals=[i.str for i in self.neg]).to(self.device)


In [31]:
emb_pos = get_embeddings_individuals(individuals=[i.str for i in pos])
emb_neg = get_embeddings_individuals(individuals=[i.str for i in neg])

In [12]:
start_class = OWLThing

In [13]:
number_of_tested_concepts = 0

In [14]:
#reward_func = CeloeBasedReward()


In [14]:
def create_rl_state(c: OWLClassExpression,parent_node: Optional[RL_State] = None,
                        is_root: bool = False) -> RL_State:
    rl_state = RL_State(c, parent_node=parent_node, is_root=is_root)
    rl_state.length = get_expression_length(c)
    return rl_state

In [15]:
quality_func = compute_f1_score_from_confusion_matrix

In [16]:
def compute_quality_of_class_expression(state: RL_State, number_of_tested_concepts=None):
    if isinstance(kb, TripleStore) and state.concept is not OWLThing:
        sparql_query = owl_expression_to_sparql_with_confusion_matrix(expression=state.concept, positive_examples = pos, negative_examples = neg)
        bindings = kb.query(sparql_query).json()["results"]["bindings"]
        assert len(bindings) == 1
        bindings = bindings.pop()
        confusion_matrix = {k: v["value"]for k, v in bindings.items()}
        quality = quality_func(confusion_matrix=confusion_matrix)

    else:
        individuals = frozenset([i for i in kb.individuals(state.concept,True)])
        quality = compute_f1_score(individuals=individuals, pos=pos, neg=neg)
    state.quality = quality
    number_of_tested_concepts += 1

In [17]:
# a function to return RL state
def initialize_training_class_expression_learning_problem(pos,neg,number_of_tested_concepts)-> RL_State:
    pos_emb = get_embeddings_individuals(individuals=[i.str for i in pos])
    neg_emb = get_embeddings_individuals(individuals=[i.str for i in neg])
    root_rl_state = create_rl_state(start_class, is_root=True)
    compute_quality_of_class_expression(root_rl_state, number_of_tested_concepts)
    return root_rl_state

In [18]:
initial_state = initialize_training_class_expression_learning_problem(pos, neg,number_of_tested_concepts)
initial_state.concept

OWLClass(IRI('http://www.w3.org/2002/07/owl#', 'Thing'))

In [19]:
refinement_operator = LengthBasedRefinement(knowledge_base = kb,
                                            use_inverse = True,
                                            use_data_properties = True,
                                            use_card_restrictions = True,
                                            use_nominals = True,
                                            min_cardinality_restriction = 2,
                                            max_cardinality_restriction = 5)

In [20]:
def apply_refinement(rl_state: RL_State) -> Generator:
        """ Downward refinements"""
        assert isinstance(rl_state, RL_State), f"It must be rl state {rl_state}"
        assert isinstance(rl_state.concept, OWLClassExpression)
        operator = refinement_operator
        for i in operator.refine(rl_state.concept):  # O(N)
            yield create_rl_state(i, parent_node=rl_state)

In [21]:
next_rl_states = list(apply_refinement( initial_state))
next_rl_states

[<class 'ontolearn.search.RL_State'> at 0x91795b	person	Quality:None	Heuristic:None	Length:1,
 <class 'ontolearn.search.RL_State'> at 0x917962	¬female	Quality:None	Heuristic:None	Length:2,
 <class 'ontolearn.search.RL_State'> at 0x917969	¬male	Quality:None	Heuristic:None	Length:2,
 <class 'ontolearn.search.RL_State'> at 0x917970	person ⊔ person	Quality:None	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x917977	person ⊔ (¬female)	Quality:None	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x91797e	person ⊔ (¬male)	Quality:None	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x917985	∃ hasChild.person	Quality:None	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x91798c	∀ hasChild.person	Quality:None	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x917993	∃ hasChild⁻.person	Quality:None	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x91799a	∀ hasChild⁻.person	Quality:None	Heuristi

In [43]:
next_next_rl_states = list(apply_refinement( next_rl_states[0]))
next_next_rl_states

[<class 'ontolearn.search.RL_State'> at 0x9977bc	person ⊓ (≥ 4 hasChild.person)	Quality:None	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x9977c3	person ⊓ (≥ 4 hasChild.(¬female))	Quality:None	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x9977ca	person ⊓ (≥ 3 hasChild.⊤)	Quality:None	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x9976a4	person ⊓ (∃ hasChild⁻.⊥)	Quality:None	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x997626	person ⊓ (≥ 2 hasChild.(¬male))	Quality:None	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x997714	person ⊓ person	Quality:None	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x9977d1	person ⊓ (≥ 3 hasChild⁻.⊥)	Quality:None	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x9977d8	person ⊓ (¬female)	Quality:None	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x9977df	person ⊓ (≥ 2 hasChild⁻.person)	Quality:None	

In [22]:
for i in next_rl_states:
    compute_quality_of_class_expression(i, number_of_tested_concepts)
next_rl_states

[<class 'ontolearn.search.RL_State'> at 0x91795b	person	Quality:0.4	Heuristic:None	Length:1,
 <class 'ontolearn.search.RL_State'> at 0x917962	¬female	Quality:0.6666666666666666	Heuristic:None	Length:2,
 <class 'ontolearn.search.RL_State'> at 0x917969	¬male	Quality:0.0	Heuristic:None	Length:2,
 <class 'ontolearn.search.RL_State'> at 0x917970	person ⊔ person	Quality:0.4	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x917977	person ⊔ (¬female)	Quality:0.4	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x91797e	person ⊔ (¬male)	Quality:0.4	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x917985	∃ hasChild.person	Quality:0.6666666666666666	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x91798c	∀ hasChild.person	Quality:0.4	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x917993	∃ hasChild⁻.person	Quality:0.0	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x91799a	∀ hasChild⁻.person	

In [44]:
for i in next_next_rl_states:
    compute_quality_of_class_expression(i, number_of_tested_concepts)
next_next_rl_states

[<class 'ontolearn.search.RL_State'> at 0x9977bc	person ⊓ (≥ 4 hasChild.person)	Quality:0.0	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x9977c3	person ⊓ (≥ 4 hasChild.(¬female))	Quality:0.0	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x9977ca	person ⊓ (≥ 3 hasChild.⊤)	Quality:0.0	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x9976a4	person ⊓ (∃ hasChild⁻.⊥)	Quality:0.0	Heuristic:None	Length:6,
 <class 'ontolearn.search.RL_State'> at 0x997626	person ⊓ (≥ 2 hasChild.(¬male))	Quality:0.0	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x997714	person ⊓ person	Quality:0.4	Heuristic:None	Length:3,
 <class 'ontolearn.search.RL_State'> at 0x9977d1	person ⊓ (≥ 3 hasChild⁻.⊥)	Quality:0.0	Heuristic:None	Length:7,
 <class 'ontolearn.search.RL_State'> at 0x9977d8	person ⊓ (¬female)	Quality:0.6666666666666666	Heuristic:None	Length:4,
 <class 'ontolearn.search.RL_State'> at 0x9977df	person ⊓ (≥ 2 hasChild⁻.person)	Qualit

#### Analyze the length, quality of the concepts generated by the refinement operator

In [23]:
concepts = []
exprs = []
lengths = []
qualities = []
for i in next_rl_states:
    concepts.append(i.concept)
    exprs.append(owl_expression_to_dl(i.concept))
    lengths.append(i.length)
    qualities.append(i.quality) 
    
# make dictionary of the keys exprs, values lengths and qualities
concept_dict = {e: {'length': l, 'quality': q} for e, l, q in zip(exprs, lengths, qualities)}

In [48]:
next_concepts = []
next_exprs = []
next_lengths = []
next_qualities = []
for i in next_next_rl_states:
    next_concepts.append(i.concept)
    next_exprs.append(owl_expression_to_dl(i.concept))
    next_lengths.append(i.length)
    next_qualities.append(i.quality)
# make dictionary of the keys exprs, values lengths and qualities
next_concept_dict = {e: {'length': l, 'quality': q} for e, l, q in zip(next_exprs, next_lengths, next_qualities)}

In [24]:
concept_dict

{'person': {'length': 1, 'quality': 0.4},
 '¬female': {'length': 2, 'quality': 0.6666666666666666},
 '¬male': {'length': 2, 'quality': 0.0},
 'person ⊔ person': {'length': 3, 'quality': 0.4},
 'person ⊔ (¬female)': {'length': 4, 'quality': 0.4},
 'person ⊔ (¬male)': {'length': 4, 'quality': 0.4},
 '∃ hasChild.person': {'length': 3, 'quality': 0.6666666666666666},
 '∀ hasChild.person': {'length': 3, 'quality': 0.4},
 '∃ hasChild⁻.person': {'length': 4, 'quality': 0.0},
 '∀ hasChild⁻.person': {'length': 4, 'quality': 0.4},
 '≥ 2 hasChild.person': {'length': 4, 'quality': 0.0},
 '≥ 2 hasChild⁻.person': {'length': 5, 'quality': 0.0},
 '≥ 3 hasChild.person': {'length': 4, 'quality': 0.0},
 '≥ 3 hasChild⁻.person': {'length': 5, 'quality': 0.0},
 '≥ 4 hasChild.person': {'length': 4, 'quality': 0.0},
 '≥ 4 hasChild⁻.person': {'length': 5, 'quality': 0.0},
 '∃ hasChild.⊤': {'length': 3, 'quality': 0.6666666666666666},
 '∀ hasChild.⊤': {'length': 3, 'quality': 0.4},
 '∃ hasChild⁻.⊤': {'length': 

In [49]:
next_concept_dict

{'person ⊓ (≥ 4 hasChild.person)': {'length': 6, 'quality': 0.0},
 'person ⊓ (≥ 4 hasChild.(¬female))': {'length': 7, 'quality': 0.0},
 'person ⊓ (≥ 3 hasChild.⊤)': {'length': 6, 'quality': 0.0},
 'person ⊓ (∃ hasChild⁻.⊥)': {'length': 6, 'quality': 0.0},
 'person ⊓ (≥ 2 hasChild.(¬male))': {'length': 7, 'quality': 0.0},
 'person ⊓ person': {'length': 3, 'quality': 0.4},
 'person ⊓ (≥ 3 hasChild⁻.⊥)': {'length': 7, 'quality': 0.0},
 'person ⊓ (¬female)': {'length': 4, 'quality': 0.6666666666666666},
 'person ⊓ (≥ 2 hasChild⁻.person)': {'length': 7, 'quality': 0.0},
 'person ⊓ (∃ hasChild.⊥)': {'length': 5, 'quality': 0.0},
 'person ⊓ (≥ 2 hasChild.⊥)': {'length': 6, 'quality': 0.0},
 'person ⊓ (≥ 4 hasChild⁻.person)': {'length': 7, 'quality': 0.0},
 'person ⊓ (≥ 2 hasChild⁻.(¬female))': {'length': 8, 'quality': 0.0},
 'person ⊓ (≥ 4 hasChild.(¬male))': {'length': 7, 'quality': 0.0},
 'person ⊓ (≥ 4 hasChild⁻.(¬female))': {'length': 8, 'quality': 0.0},
 'person ⊓ (person ⊔ person)': {'l

#### Batch Weighted A*/Q* Search (BWAS) with DNN/DQN Heuristic

In [ ]:
# g(concept) = path cost to reach the concept from the start concept (OWLThing)
# h(concept, goal) =  is the heuristic value, the estimated cost-to-go from the state associated with concept to a nearest goal state [heuristic function (e.g., using a DNN/DQN)]
# BWAS is a generalization of A* search since A* search can be recovered by setting λ to 1 and B to 1, where λ is the weight given to the heuristic function and B is the batch size.
"""
OPEN = priority queue of nodes based on minimal f
CLOSED = maps states to their shortest discovered path costs
UB, nUB = inf, NILL
LB = 0
INITIALIZE:
initial_state = OWLThing
n_initial_state = NODE(initial_state, g=0, h=DQNHeuristic(initial_state, goal), f=g+h)
INSERT n_initial_state INTO OPEN

WHILE OPEN is not empty DO:
    generated = []
    WHILE |generated| < B AND OPEN is not empty DO:
        current_state = EXTRACT_MIN(OPEN)
        if generated is empty THEN 
            LB = max(LB, f(current_state))
        if current_state is goal THEN [quality >= threshold]
            if UB > g(current_state) THEN
                UB, state_with_UB = g(current_state), current_state
                continue        
        for action in available_actions[in the refinement operators]
        next states = list(apply_refinement( current_state))
        g(current_state) = path cost to reach the current state from the start state
        for next_state in next_states: 
            h(next_state, goal) = heuristic value predicted by DQN/DNN model 
            g(next_state) = g(current_state) + h(next_state, goal) # λ to 1 for now
            if next_state not in CLOSED OR g(next_state) < CLOSED[next_state] THEN
            next_state.g = g(next_state)
            generted.append(next_state)
        if LB >= UB THEN
            RETURN state_with_UB ? path to state_with_UB ?
        generated_states = get_states(generated)
        heuristic_values = DQNHeuristic(generated_states)
        for 0 <= j <= |generated| DO:
            state, g = generated[j], generated[j].g
            state.h = heuristic_values[j]
            PUSH state INTO OPEN with priority f = g + h
    return state_with_UB ? path to state_with_UB ?           
    
"""
class PriorityQueue:
    """Simple min-heap wrapper with (priority, counter, item)."""
    def __init__(self):
        self.heap = []
        self.counter = 0

    def push(self, item, priority):
        heapq.heappush(self.heap, (priority, self.counter, item))
        self.counter += 1

    def pop(self):
        if not self.heap:
            return None
        return heapq.heappop(self.heap)[2]

    def empty(self):
        return len(self.heap) == 0


def BWAS(initial_state,
         refinement_operator,
         heuristic,
         batch_size: int = 10,
         quality_threshold: float = 1.0):
    OPEN = PriorityQueue()
    CLOSED = {}
    UB, state_with_UB = float('inf'), None
    LB = 0.0
    # Initialize
    initial_rl_state = initial_state
    initial_rl_state.g = 0.0
    heuristic.apply(initial_rl_state)
    OPEN.push(initial_rl_state, initial_rl_state.g + initial_rl_state.heuristic)
    while not OPEN.empty():
        generated = []
        while len(generated) < batch_size and not OPEN.empty():
            current_state = OPEN.pop()
            if not generated:
                LB = max(LB, current_state.g + current_state.heuristic)
            if current_state.quality >= quality_threshold:
                if UB > current_state.g:
                    UB, state_with_UB = current_state.g, current_state
                    continue
            next_states = list(refinement_operator.refine(current_state))
            for next_state in next_states:
                g_next = current_state.g + 1.0  
                if next_state not in CLOSED or g_next < CLOSED[next_state]:
                    next_state.g = g_next
                    generated.append(next_state)
        if LB >= UB:
            return state_with_UB
        for state in generated:
            heuristic.apply(state)
            OPEN.push(state, state.g + state.heuristic)
    return state_with_UB

In [126]:
# quality based BWAS search with DNN heuristic 
""" 
How to use quality and length as cost here?
In A* search, f(c) = g(c) + h(c)
Let's say g(c) is related to length of the concept and h(c) is related to predicted quality to reach goal.
For a concept c, 
    - length(c) = c.length, 
      max_expr_length = 20, 
      normalized_length(c) = length(c) / max_expr_length
      g = min(1, normalized_length(c))  # between 0 and 1
    - quality(c) = DNN(emb_c.squeeze(1), emb_goal.squeeze(1)) # predicted quality between 0 and 1
      h = 1 - quality(c)  # between 0 and 1  
    - f(c) = 0.2*g + 0.8*h  
    - How does minimize cost f works here?
      concept c1 | length=5, max_length=20, g = 0.25 | quality = 1.0, h=0.0 | f = 0.05 (cost low, good)
      concept c2 | length=5, max_length=20, g = 0.25 | quality = 0.2, h=0.8 | f = 0.65 (cost high, bad)
      concept c3 | length=10, max_length=20, g = 0.5 | quality = 1.0, h=0.0 | f = 0.1 (cost low, but not best)
"""

class PriorityQueue:
    """Min-heap with (priority, counter, item)."""
    def __init__(self):
        self.heap = []
        self.counter = 0

    def push(self, item, priority):
        heapq.heappush(self.heap, (priority, self.counter, item))
        self.counter += 1

    def pop(self):
        return None if not self.heap else heapq.heappop(self.heap)[2]

    def empty(self):
        return len(self.heap) == 0

 
# A* PRIORITY: use g = Length, h = Heuristic 
def get_f(state, lambda_g=0.5, lambda_h=0.5):
    g = getattr(state, "length", 0)
    h = getattr(state, "heuristic", 0)
    if h is None:
        h = 0
    return lambda_g * g + lambda_h * h

 
def quality_BWAS(
    initial_state, 
    DNN,
    emb_goal, 
    batch_size: int = 10,
    quality_threshold: float = 0.99
):
    """
    BWAS using ONLY RL_State built-in fields:
      - state.length      (g cost)
      - state.heuristic   (distance-to-go heuristic)
      - state.quality     (quality from 0 to 1)

    If Quality not set → compute via DNN
    Heuristic is always computed as (1 - Quality)
    """

    OPEN = PriorityQueue()
    CLOSED = {}     # direct RL_State object → best Length seen
    UB = float("inf")
    LB = 0.0
    best_state = None

    # compute quality via DNN 
    def compute_quality(state):
        if getattr(state, "quality", None) is not None:
            return state.quality
        emb_state = get_embeddings_individuals(
            [ind.str for ind in kb.individuals(state.concept, True)]
        )
        q = float(DNN(emb_state.squeeze(1), emb_goal.squeeze(1)).item())
        state.quality = q
        return q

    # Initialize 
    q0 = compute_quality(initial_state)
    h0 = 1.0 - q0
    initial_state.heuristic = h0

    f0 = get_f(initial_state)
    OPEN.push(initial_state, f0)
    CLOSED[initial_state] = initial_state.length

    # MAIN LOOP
    while not OPEN.empty():

        generated = []

        # Expand up to batch size
        while len(generated) < batch_size and not OPEN.empty():
            print(f"OPEN size: {len(OPEN.heap)}, LB: {LB}, UB: {UB}, Generated: {len(generated)}")

            current = OPEN.pop()
            current_h = getattr(current, "heuristic", None)
            if current_h is None:
                # recompute h from quality
                q = compute_quality(current)
                current_h = 1.0 - q
                current.heuristic = current_h

            # update LB using the first popped item
            if not generated:
                LB = max(LB, current_h)

            # GOAL CHECK — quality high enough
            cur_quality = getattr(current, "quality", None)
            if cur_quality is None:
                cur_quality = compute_quality(current)

            if cur_quality >= quality_threshold:
                if current_h < UB:
                    UB = current_h
                    best_state = current
                continue

            # refine
            next_states = apply_refinement(current)
            for nxt in next_states:
                g_next = getattr(nxt, "length", None)
                if g_next is None:
                    g_next = 999   # fallback (very large cost)

                # prune if worse g for same object
                if nxt in CLOSED and g_next >= CLOSED[nxt]:
                    continue

                CLOSED[nxt] = g_next
                generated.append(nxt)

        # termination
        if LB >= UB:
            return best_state

        # Evaluate all generated states 
        for s in generated:
            q = compute_quality(s)
            h = 1.0 - q
            s.heuristic = h

            f = get_f(s)
            OPEN.push(s, f)

    return best_state

In [127]:
initial_state = initialize_training_class_expression_learning_problem(pos, neg,number_of_tested_concepts)
initial_state

<class 'ontolearn.search.RL_State'> at 0x244d23	⊤	Quality:0.4	Heuristic:None	Length:1

In [128]:
emb_goal = get_embeddings_individuals(individuals=[i.str for i in pos])

In [129]:
# Step 2: get goal embedding
#goal_concept = "person ⊓ (≥ 4 hasChild.person)"
#goal_individuals = [ind.str for ind in kb.individuals(goal_concept, True)]
#emb_goal = get_embeddings_individuals(goal_individuals)  # torch tensor [B,1,D]

# Step 3: call quality_BWAS

best_state = quality_BWAS(
    initial_state=initial_state, 
    DNN=dnn_model,
    emb_goal=emb_goal 
)

# Step 4: inspect result
print("Best state concept:", owl_expression_to_dl(best_state.concept))
print("Length:", best_state.length)
print("Predicted quality:", best_state.quality)
print("Heuristic:", best_state.heuristic)


OPEN size: 1, LB: 0.0, UB: inf, Generated: 0
OPEN size: 56, LB: 0.6, UB: inf, Generated: 0
OPEN size: 111, LB: 0.6, UB: inf, Generated: 0
OPEN size: 110, LB: 0.6, UB: inf, Generated: 2
OPEN size: 109, LB: 0.6, UB: inf, Generated: 4
OPEN size: 173, LB: 0.6, UB: inf, Generated: 0
OPEN size: 172, LB: 0.9163273721933365, UB: inf, Generated: 1
OPEN size: 171, LB: 0.9163273721933365, UB: inf, Generated: 2
OPEN size: 233, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 293, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 405, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 467, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 529, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 639, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 701, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 759, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 815, LB: 0.9163273721933365, UB: inf, Generated: 0
OPEN size: 874, LB: 0.9163273721933365,

#### Define the DQN model and heuristic

In [30]:
class DNN(torch.nn.Module):
    """
        A neural model for cost Learning.
    
        An input Drill has the following form:
                1. Embedding of the current state (s). [mean of embeddings of individuals in s]
                2. Embedding of the goal state (G). [mean of embeddings of individuals in G] 
                
        Given such input, a score is predicted representing the cost to reach the goal from the current state.
        The score is used as heuristic value in BWAS and calculated as:
            h(s, G) = DNN(emb(s), emb(G)) 
            Internal representation:
                X = [emb(s),
                     emb(G),
                     emb(s) - emb(G),
                     emb(s) * emb(G)]  # 4 x D tensor
            Architecture:
                Conv1D(4 → 32)
                ReLU
                Conv1D(32 → 32)
                ReLU
                GlobalMaxPool
                Linear(32 → 1)
    """
    def __init__(self,embedding_dim=256, hidden_channels=32): 
        super(DNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_channels = hidden_channels
        # Conv1D expects input shape: (batch, channels, seq_length)
        self.conv1 = torch.nn.Conv1d(in_channels=4,
                                     out_channels=self.hidden_channels,
                                     kernel_size=3,
                                     padding=1, stride=1, bias=True)
        self.conv2 = torch.nn.Conv1d(in_channels=self.hidden_channels,
                                        out_channels=self.hidden_channels,
                                        kernel_size=3,
                                        padding=1, stride=1, bias=True) 
        self.fc = torch.nn.Linear(in_features=self.hidden_channels, out_features=1)
        self.activation = torch.nn.ReLU()
    
    def forward(self, emb_s, emb_g):
        """
        emb_s:  batch_size x 1 x embedding_dim
        emb_g:  batch_size x 1 x embedding_dim
        """
        batch_size = emb_s.size(0)
        emb_diff = emb_s - emb_g    
        emb_prod = emb_s * emb_g
        X = torch.stack([emb_s, emb_g, emb_diff, emb_prod], dim=1)
        # X: batch_size x 4 x embedding_dim
        X = self.activation(self.conv1(X))
        X = self.activation(self.conv2(X))
        X = torch.max(X, dim=2).values
        scores = self.fc(X).squeeze()
        return scores      
        
class DNNHeuristic:
    def __init__(self, model: DNN, device: torch.device):
        self.model = model
        self.device = device
        self.model.eval()
        def score(self, node, parent_node=None):
            """ Compute heuristic value of root node only"""
            if parent_node is None and node.is_root:
                return torch.FloatTensor([.0001], device=self.device).squeeze()
            raise ValueError

        def apply(self, node, parent_node=None):
            """ Assign predicted Q-value to node object."""
            predicted_q_val = self.score(node, parent_node)
            node.heuristic = predicted_q_val

In [ ]:
# train DNN model here to predict quality values
"""
data is next_rl_states
emb_g = emb_pos
for each state in data:
emb_s = get_embeddings_individuals(individuals=[ind.str for ind in kb.individuals(state.concept,True)])
labels = state.quality
now train the DNN model with (emb_s, emb_g) as input and labels as target
"""

In [93]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dnn_model = DNN().to(device)
dnn_heuristic = DNNHeuristic(model=dnn_model, device=device)
# make data for training the DNN model
train_data = []
for state in next_rl_states:
    emb_s = get_embeddings_individuals(individuals=[ind.str for ind in kb.individuals(state.concept,True)]).to(device)
    emb_g = emb_pos.to(device)
    label = torch.FloatTensor([state.quality]).to(device)
    train_data.append((emb_s, emb_g, label))

In [94]:
# see train data shape for first element
train_data[0][0].shape, train_data[0][1].shape, train_data[0][2].shape

(torch.Size([1, 1, 256]), torch.Size([1, 1, 256]), torch.Size([1]))

In [95]:
# test data with next_next_rl_states
test_data = []
for state in next_next_rl_states:
    emb_s = get_embeddings_individuals(individuals=[ind.str for ind in kb.individuals(state.concept,True)]).to(device)
    emb_g = emb_pos.to(device)
    label = torch.FloatTensor([state.quality]).to(device)
    test_data.append((emb_s, emb_g, label))

In [96]:
for epoch in range(10):  # number of epochs
    for emb_s, emb_g, label in train_data:
        dnn_model.train()
        optimizer = torch.optim.Adam(dnn_model.parameters(), lr=0.001)
        criterion = torch.nn.MSELoss()
        optimizer.zero_grad()
        output = dnn_model(emb_s.squeeze(1), emb_g.squeeze(1))
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

C:\Users\Pritilata\miniconda3\envs\ontolearn\lib\site-packages\torch\nn\modules\loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [97]:
loss

tensor(1.2142e-05, grad_fn=<MseLossBackward0>)

In [98]:
# eval the DNN model on train data
dnn_model.eval()
with torch.no_grad():
    for emb_s, emb_g, label in train_data:
        output = dnn_model(emb_s.squeeze(1), emb_g.squeeze(1))
        print(f"Predicted: {output.item()}, Actual: {label.item()}")

Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.7118064165115356, Actual: 0.6666666865348816
Predicted: 0.08010192215442657, Actual: 0.0
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.8142462968826294, Actual: 0.6666666865348816
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.06590015441179276, Actual: 0.0
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.13948094844818115, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.8142462968826294, Actual: 0.6666666865348816
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.06590015441179276, Actual: 0.0
Predicted: 0.50030380487

In [99]:
# eval the DNN model on test data
dnn_model.eval()
with torch.no_grad():
    for emb_s, emb_g, label in test_data:
        output = dnn_model(emb_s.squeeze(1), emb_g.squeeze(1))
        print(f"Predicted: {output.item()}, Actual: {label.item()}")

Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.7118064165115356, Actual: 0.6666666865348816
Predicted: 0.13948094844818115, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.7977250814437866, Actual: 0.6666666865348816
Predicted: 0.08367262780666351, Actual: 0.0
Predicted: 0.5003038048744202, Actual: 0.4000000059604645
Predicted: 0.06590015441179276, Actual: 0.0
Predicted: 0.13948094844818115, Actual: 0.0
Predic

#### A RL agent using DNN and BWAS to find high-quality class expressions

In [ ]:
"""
What does the RL agent do here? 

Exploration Phase (True RL Learning Phase)
    - Start from a root concept, typically ⊤ (OWLThing).
    - Apply the refinement operator to generate new class expressions.
    - For each generated class expression:
        i) Compute its true quality 
        ii) Extract embeddings of the individuals satisfying that expression.
        iii) Store a training sample: (embedding_of_concept, embedding_of_goal, true_quality)
    - The agent chooses a mixture of:
        Exploration: randomly selecting concepts to evaluate.
        Exploitation: selecting concepts predicted to be good by the current DNN. [feedback loop]??
        (quality_BWAS). 
    - Periodically, the DNN is trained on accumulated samples so it improves its ability to predict quality.
    - This exploration loop continues for many episodes until:
        i) the DNN converges, or
        ii) reach a fixed number of iterations.
    - At the end of exploration, the DNN acts as a value function that estimates the quality of a class expression. It becomes the learned heuristic.
Exploitation Phase (BWAS with DNN-guided Search)
    - Search for the highest-quality class expression using A*-style cost.
    - Start from the initial concept (⊤).
    - get the best class expression using quality_BWAS function with the trained DNN as heuristic.
"""

#### Define the DQN model and heuristic

In [ ]:
# The Neural Lookahead (DQN)
class DQN(torch.nn.Module):
    def __init__(self): 
        super(DQN, self).__init__()
        self.embedding_dim = 256
        

In [ ]:
class DQNHeuristic:
    def __init__(self, model: DQN, device: torch.device):
        self.model = model
        self.device = device
        self.model.eval()
        def score(self, node, parent_node=None):
            """ Compute heuristic value of root node only"""
            if parent_node is None and node.is_root:
                return torch.FloatTensor([.0001], device=self.device).squeeze()
            raise ValueError

        def apply(self, node, parent_node=None):
            """ Assign predicted Q-value to node object."""
            predicted_q_val = self.score(node, parent_node)
            node.heuristic = predicted_q_val